# Skill 4 — Project Repository, Git Workflow and PyTorch–CUDA Environment Setup

Run this notebook in **Google Colab**. It creates a sample project repository, demonstrates Git initialisation/branches/commits, checks Python and PyTorch, detects GPU/CUDA/cuDNN availability, runs a matrix multiplication test, and writes an environment report.

**Before running:** In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU) if you want to test GPU execution. A GPU is not guaranteed to be available in every Colab session.

In [ ]:
# 1. Check Python and system details
import sys, platform, os, subprocess
print("Python:", sys.version)
print("Platform:", platform.platform())

def run_command(command):
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    print(result.stdout if result.stdout else result.stderr)
    return result.returncode

print("\nGit version:")
run_command("git --version")

print("\nNVIDIA GPU information (if available):")
gpu_check = subprocess.run("nvidia-smi", shell=True, text=True, capture_output=True)
if gpu_check.returncode == 0:
    print(gpu_check.stdout)
else:
    print("nvidia-smi is unavailable. Check that a GPU runtime is enabled in Colab.")

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39

Git version:
git version 2.43.0


NVIDIA GPU information (if available):
nvidia-smi is unavailable. Check that a GPU runtime is enabled in Colab.


In [ ]:
# 2. Create the prescribed project repository structure
from pathlib import Path

PROJECT = Path("/content/DL-Skill-4")
folders = ["src", "data", "models", "notebooks", "results", "reports"]

PROJECT.mkdir(parents=True, exist_ok=True)
for folder in folders:
    (PROJECT / folder).mkdir(exist_ok=True)

(PROJECT / "README.md").write_text(
    "# DL Skill 4\nProject repository and PyTorch-CUDA environment setup.\n"
)
(PROJECT / ".gitignore").write_text(
    ".venv/\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n"
)

print("Created repository:", PROJECT)
for path in sorted(PROJECT.rglob("*")):
    print(path.relative_to(PROJECT))

Created repository: /content/DL-Skill-4
.gitignore
README.md
data
models
notebooks
reports
results
src


In [ ]:
# 3. Check PyTorch, CUDA, cuDNN and GPU visibility
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version used by PyTorch:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("cuDNN enabled:", torch.backends.cudnn.enabled)

gpu_name = "Not available"
gpu_count = torch.cuda.device_count()

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU count:", gpu_count)
    print("GPU name:", gpu_name)
else:
    print("No CUDA GPU detected. The CPU test will still run.")

PyTorch version: 2.14.1+cpu
CUDA available: False
CUDA version used by PyTorch: None
cuDNN version: None
cuDNN enabled: True
No CUDA GPU detected. The CPU test will still run.


In [ ]:
# 4. Run a matrix multiplication test on GPU when available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
a = torch.rand((512, 512), device=device)
b = torch.rand((512, 512), device=device)
c = torch.matmul(a, b)

if device.type == "cuda":
    torch.cuda.synchronize()

print("Execution device:", device)
print("Result shape:", tuple(c.shape))
print("First 3 values:", c.flatten()[:3].cpu().tolist())
print("Test successful:", c.shape == (512, 512))

Execution device: cpu
Result shape: (512, 512)
First 3 values: [125.47856903076172, 123.36019134521484, 127.1619644165039]
Test successful: True


In [ ]:
# 5. Record installed package versions
import importlib.metadata as metadata

packages = ["torch", "torchvision", "torchaudio", "numpy", "pandas", "matplotlib", "jupyter"]
requirements_lines = []
for package in packages:
    try:
        version = metadata.version(package)
        requirements_lines.append(f"{package}=={version}")
        print(f"{package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

(PROJECT / "reports" / "requirements.txt").write_text("\n".join(requirements_lines) + "\n")

torch: 2.14.1+cpu
torchvision: 0.29.1+cpu
torchaudio: 2.11.0+cpu
numpy: 2.4.6
pandas: 2.2.3
matplotlib: 3.10.0
jupyter: not installed


111

In [ ]:
# 6. Demonstrate Git repository, branch and commit workflow
def git(command):
    result = subprocess.run(
        command, cwd=PROJECT, shell=True, text=True, capture_output=True
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Git command failed: {command}")

# Initialise Git and configure a local identity for this Colab demonstration
if not (PROJECT / ".git").exists():
    git("git init")
git('git config user.name "Colab Student"')
git('git config user.email "student@example.com"')

git("git add .")
# Avoid failing if there is nothing new to commit
status = subprocess.run("git status --porcelain", cwd=PROJECT, shell=True, capture_output=True, text=True)
if status.stdout.strip():
    git('git commit -m "Initial project structure and environment setup"')

# Create/switch to a development branch
branches = subprocess.run("git branch --list development", cwd=PROJECT, shell=True, capture_output=True, text=True)
if branches.stdout.strip():
    git("git checkout development")
else:
    git("git checkout -b development")

# Add the environment check result as a second commit
(PROJECT / "results" / "environment-check.txt").write_text(
    f"PyTorch: {torch.__version__}\n"
    f"CUDA available: {torch.cuda.is_available()}\n"
    f"CUDA version: {torch.version.cuda}\n"
    f"cuDNN version: {torch.backends.cudnn.version()}\n"
    f"GPU: {gpu_name}\n"
    f"Test device: {device}\n"
)
git("git add .")
status = subprocess.run("git status --porcelain", cwd=PROJECT, shell=True, capture_output=True, text=True)
if status.stdout.strip():
    git('git commit -m "Add environment verification results"')

print("\nGit history:")
git("git --no-pager log --oneline --decorate -5")
print("\nCurrent branch:")
git("git branch --show-current")

Initialized empty Git repository in /content/DL-Skill-4/.git/
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
[master (root-commit) fb8cab7] Initial project structure and environment setup
 3 files changed, 12 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 reports/requirements.txt
Switched to a new branch 'development'
[development 7ddf638] Add environment verification results
 1 file changed, 6 insertions(+)
 create mode 100644 results/environment-check.txt

Git history:
7ddf638 (HEAD -> development) Add envi

In [ ]:
# 7. Generate the environment report
report = f\'''# Deep Learning Environment Setup Report

## System Information
- Platform: {platform.platform()}
- Python: {sys.version.split()[0]}
- GPU: {gpu_name}
- GPU count: {gpu_count}

## Framework Information
- PyTorch: {torch.__version__}
- CUDA available: {torch.cuda.is_available()}
- CUDA version (PyTorch): {torch.version.cuda}
- cuDNN version: {torch.backends.cudnn.version()}
- cuDNN enabled: {torch.backends.cudnn.enabled}

## Verification
- PyTorch imported successfully: Yes
- Matrix multiplication test: Successful
- Execution device: {device}

## Conclusion
The project repository and folder structure were created.
Git initialisation, branch creation and commits were demonstrated.
PyTorch and CUDA/GPU availability were checked, and a matrix
multiplication operation was executed on the available device.
\'''

report_path = PROJECT / "reports" / "environment-report.md"
report_path.write_text(report)
print(report)

SyntaxError: unexpected character after line continuation character (3431999981.py, line 2)

In [ ]:
# 8. Package the project folder as a ZIP file for download
import shutil
zip_path = shutil.make_archive("/content/DL-Skill-4", "zip", root_dir="/content", base_dir="DL-Skill-4")
print("Created:", zip_path)

from google.colab import files
files.download(zip_path)

## Optional: connect this repository to your GitHub account

The notebook demonstrates Git locally. To publish it, create an empty GitHub repository, then run the commands below after replacing the URL with your repository URL. GitHub authentication may be required.

```bash
%cd /content/DL-Skill-4
!git remote add origin https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git
!git push -u origin main
!git push -u origin development
```

If your initial branch is not named `main`, check it with `!git branch --show-current` and push the correct branch.

**Note:** Colab provides a temporary runtime. Download the ZIP or push the repository to GitHub before ending the session.